<a href="https://colab.research.google.com/github/aronwilbert/COMP8420-Group-L-Healthcare/blob/main/COMP8420_Group_L_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from datasets import load_dataset, DatasetDict
import pandas as pd

In [12]:
# 1. Load the raw master dataset
master_dataset = load_dataset("xDAN-datasets/ChatDoctor_HealthCareMagic_112k", split="train")
print(f"📊 Initial Raw Master Dataset Samples: {len(master_dataset)}")

# --- VISUAL 1: DISPLAY RAW DATASET BEFORE CLEANING ---
print("\n--- 🛑 RAW DATASET PREVIEW (BEFORE CLEANING) ---")
df_raw_preview = pd.DataFrame(master_dataset.select(range(5)))
display(df_raw_preview)

# 2. Filter out rows where 'input' or 'output' are missing or completely empty
clean_master = master_dataset.filter(
    lambda x: x["input"] is not None and x["output"] is not None and
              str(x["input"]).strip() != "" and str(x["output"]).strip() != ""
)
print(f"\n🧹 Cleaned Master Dataset Samples (After NA/Blank Removal): {len(clean_master)}")
print(f"❌ Total Defective Rows Removed: {len(master_dataset) - len(clean_master)}")

# 3. Securely isolate EXACTLY 5,000 clean samples for the project
dataset = clean_master.select(range(5000))
print(f"✅ Final Target Samples Isolated for Training/Analysis: {len(dataset)}")

# --- VISUAL 2: DISPLAY FINAL DATASET AFTER CLEANING ---
df_clean_final = pd.DataFrame({
    'input': [row['input'] for row in dataset],
    'output': [row['output'] for row in dataset]
})

print("\n--- 📋 VERIFIED CLEAN DATASET SNAPSHOT (AFTER CLEANING) ---")
display(df_clean_final.head(5))

📊 Initial Raw Master Dataset Samples: 112165

--- 🛑 RAW DATASET PREVIEW (BEFORE CLEANING) ---


,conversations,input,output
0,"[{'from': 'human', 'value': 'I woke up this mo...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"[{'from': 'human', 'value': 'My baby has been ...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"[{'from': 'human', 'value': 'Hello, My husband...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"[{'from': 'human', 'value': 'lump under left n...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"[{'from': 'human', 'value': 'I have a 5 month ...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...



🧹 Cleaned Master Dataset Samples (After NA/Blank Removal): 112156
❌ Total Defective Rows Removed: 9
✅ Final Target Samples Isolated for Training/Analysis: 5000

--- 📋 VERIFIED CLEAN DATASET SNAPSHOT (AFTER CLEANING) ---


,input,output
0,I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [17]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from IPython.display import display_html

# Ensure your dataset is ready as a list of strings
raw_documents = df_clean_final['input'].astype(str).tolist()

# =====================================================================
# 📋 PHASE 1: PRE-CLEANING DIAGNOSTIC PROFILE
# =====================================================================
print("=== 📋 PHASE 1: PRE-CLEANING VOCABULARY PROFILE ===")

# 1. Pre-Clean TF-IDF
pre_tfidf = TfidfVectorizer(stop_words='english', max_features=10)
pre_tfidf_matrix = pre_tfidf.fit_transform(raw_documents)
df_pre_tfidf = pd.DataFrame({'Pre-Clean Word': pre_tfidf.get_feature_names_out(), 'TF-IDF Score': pre_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

# 2. Pre-Clean N-Grams (Phrases)
pre_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10)
pre_count_matrix = pre_count.fit_transform(raw_documents)
df_pre_ngrams = pd.DataFrame({'Pre-Clean Phrase': pre_count.get_feature_names_out(), 'Frequency Count': pre_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

# Format and Display Side-by-Side with Titles
html_pre = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Individual Word Importance (TF-IDF)</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Multi-Word Phrases (N-grams / Bag-of-Words)</h3>
        {}
    </div>
</div>
""".format(df_pre_tfidf.to_html(index=False), df_pre_ngrams.to_html(index=False))
display_html(html_pre, raw=True)


# =====================================================================
# 🧼 PHASE 2: REGEX CLEANING EXECUTION
# =====================================================================
print("\n" + "="*80 + "\n=== 🧼 PHASE 2: EXECUTING REGEX CLEANING PIPELINE ===")

def meticulous_medical_cleaner(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'^\b(hi+|hello+|hey|dear)\b\s*\b(doc|doctor)?\b[,\s\.]*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(thank\s*you|thanks)\s*\b(doc|doctor)?\b[!\.]*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d+\s*(years?|yrs?|months?|mos?|mths?)\s*(old)?\b', '[REDACTED_AGE]', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

df_clean_final['clean_input'] = df_clean_final['input'].apply(meticulous_medical_cleaner)
clean_documents = df_clean_final['clean_input'].tolist()

for i in range(5):
    print(f"\n👉 ROW {i} VALIDATION:")
    print(f"   [Raw]  : {df_clean_final['input'].iloc[i][:110]}...")
    print(f"   [Clean]: {df_clean_final['clean_input'].iloc[i][:110]}...")


# =====================================================================
# ✅ PHASE 3: POST-CLEANING VERIFICATION PROFILE
# =====================================================================
print("\n" + "="*80 + "\n=== ✅ PHASE 3: POST-CLEANING VERIFICATION PROFILE ===")

# 1. Post-Clean TF-IDF
post_tfidf = TfidfVectorizer(stop_words='english', max_features=10)
post_tfidf_matrix = post_tfidf.fit_transform(clean_documents)
df_post_tfidf = pd.DataFrame({'Post-Clean Word': post_tfidf.get_feature_names_out(), 'TF-IDF Score': post_tfidf_matrix.mean(axis=0).A1}).sort_values(by='TF-IDF Score', ascending=False).reset_index(drop=True)

# 2. Post-Clean N-Grams (Phrases)
post_count = CountVectorizer(stop_words='english', ngram_range=(2, 3), max_features=10)
post_count_matrix = post_count.fit_transform(clean_documents)
df_post_ngrams = pd.DataFrame({'Post-Clean Phrase': post_count.get_feature_names_out(), 'Frequency Count': post_count_matrix.sum(axis=0).A1}).sort_values(by='Frequency Count', ascending=False).reset_index(drop=True)

# Format and Display Side-by-Side with Titles
html_post = """
<div style="display: flex;">
    <div style="margin-right: 50px;">
        <h3>📊 Individual Word Importance (TF-IDF)</h3>
        {}
    </div>
    <div>
        <h3>🏷️ Multi-Word Phrases (N-grams / Bag-of-Words)</h3>
        {}
    </div>
</div>
""".format(df_post_tfidf.to_html(index=False), df_post_ngrams.to_html(index=False))
display_html(html_post, raw=True)

=== 📋 PHASE 1: PRE-CLEANING VOCABULARY PROFILE ===


Pre-Clean Word,TF-IDF Score
pain,0.188483
hi,0.136012
old,0.131301
like,0.129531
doctor,0.123912
days,0.119800
years,0.119387
just,0.116030
ago,0.098343
right,0.097079



=== 🧼 PHASE 2: EXECUTING REGEX CLEANING PIPELINE ===

👉 ROW 0 VALIDATION:
   [Raw]  : I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walk...
   [Clean]: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walk...

👉 ROW 1 VALIDATION:
   [Raw]  : My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are ve...
   [Clean]: My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are ve...

👉 ROW 2 VALIDATION:
   [Raw]  : Hello, My husband is taking Oxycodone due to a broken leg/surgery. He has been taking this pain medication for...
   [Clean]: My husband is taking Oxycodone due to a broken leg/surgery. He has been taking this pain medication for one mo...

👉 ROW 3 VALIDATION:
   [Raw]  : lump under left nipple and stomach pain (male) Hi,I have recently noticed a few weeks ago a lu

Post-Clean Word,TF-IDF Score
redacted_age,0.230689
pain,0.179547
like,0.127874
days,0.119208
just,0.114804
doctor,0.106497
ago,0.096186
right,0.095093
time,0.095007
left,0.081199
